In [2]:
pip install ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 1.0 MB/s eta 0:00:01
   -------------------------- ------------- 0.8/1.2 MB 1.0 MB/s eta 0:00:01
   ---------------------------------- ----- 1.0/1.2 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.0 MB/s  0:00:01
   ---------------------------------------- 0.0/823.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/823.9 kB ? eta -:--:--
   ------------------------- -------------- 524.3/823.9 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 823.9/823.9 kB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/47.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/47.0 

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

# Load YOLO model
model = YOLO("yolov8n.pt")  # lightweight model

# Function to detect traffic light color
def detect_traffic_light_color(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define color ranges
    red_lower = np.array([0, 120, 70])
    red_upper = np.array([10, 255, 255])

    yellow_lower = np.array([15, 150, 150])
    yellow_upper = np.array([35, 255, 255])

    green_lower = np.array([40, 50, 50])
    green_upper = np.array([90, 255, 255])

    # Masks
    red_mask = cv2.inRange(hsv, red_lower, red_upper)
    yellow_mask = cv2.inRange(hsv, yellow_lower, yellow_upper)
    green_mask = cv2.inRange(hsv, green_lower, green_upper)

    # Count pixels
    red_pixels = cv2.countNonZero(red_mask)
    yellow_pixels = cv2.countNonZero(yellow_mask)
    green_pixels = cv2.countNonZero(green_mask)

    if red_pixels > yellow_pixels and red_pixels > green_pixels:
        return "RED"
    elif yellow_pixels > red_pixels and yellow_pixels > green_pixels:
        return "YELLOW"
    elif green_pixels > red_pixels and green_pixels > yellow_pixels:
        return "GREEN"
    else:
        return "UNKNOWN"


# Load image
image_path = "traffic image.jpg"
image = cv2.imread(image_path)

# Run detection
results = model(image)[0]

car_count = 0
bike_count = 0
traffic_light_color = "Not Detected"

# Class IDs from COCO dataset
for box in results.boxes:
    cls = int(box.cls[0])
    label = model.names[cls]

    x1, y1, x2, y2 = map(int, box.xyxy[0])

    # Count vehicles
    if label == "car":
        car_count += 1
    elif label == "motorcycle":
        bike_count += 1

    # Traffic light detection
    elif label == "traffic light":
        tl_crop = image[y1:y2, x1:x2]
        traffic_light_color = detect_traffic_light_color(tl_crop)

# Output results
print("🚗 Cars:", car_count)
print("🏍️ Bikes:", bike_count)
print("🚦 Traffic Light Color:", traffic_light_color)

# Show image
cv2.imshow("Result", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Dell\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

0: 480x640 8 persons, 4 cars, 1 truck, 103.8ms
Speed: 7.5ms preprocess, 103.8ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
🚗 Cars: 4
🏍️ Bikes: 0
🚦 Traffic Light Color: Not Detected
